In [ ]:
import os, sys
print(sys.version, '| PYTHONPATH =', os.environ.get('PYTHONPATH'))

In [ ]:
import json
from pathlib import Path
SHARD = 1
N_SHARDS = 2
BUDGET_H = 9.0
WORKER_SRC = "\"\"\"Validate one SWE-rebench task in a plain online environment (used by the G7 spike notebook; dry-run locally first).\n\nrun_task(task, work_root) -> dict with a verdict:\n  valid             FAIL_TO_PASS all fail/error before the gold patch and all pass after; PASS_TO_PASS all pass after\n  p2p_partial       FAIL_TO_PASS behave, but some PASS_TO_PASS tests do not pass after (usable with those tests dropped)\n  f2p_passes_before at least one FAIL_TO_PASS test already passes with only the test patch (task does not test the fix)\n  f2p_fails_after   the gold patch does not make FAIL_TO_PASS pass in this environment\n  install_failed / clone_failed / patch_apply_failed / no_tests_ran / timeout\n\"\"\"\n\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nALIASES = {\"yaml\": \"pyyaml\", \"PIL\": \"pillow\", \"cv2\": \"opencv-python-headless\", \"sklearn\": \"scikit-learn\", \"dateutil\": \"python-dateutil\", \"bs4\": \"beautifulsoup4\", \"attr\": \"attrs\", \"OpenSSL\": \"pyopenssl\", \"jwt\": \"pyjwt\", \"git\": \"gitpython\", \"magic\": \"python-magic\", \"serial\": \"pyserial\", \"usb\": \"pyusb\"}\nSTATUS = re.compile(r\"^(PASSED|FAILED|ERROR|SKIPPED|XFAIL|XPASS)\\s+(.+?)(?:\\s+-\\s+.*)?$\", re.M)\n\n\ndef _run(cmd, cwd, timeout, env, shell=False, inp=None):\n    t0 = time.time()\n    try:\n        p = subprocess.run(cmd, cwd=cwd, env=env, shell=shell, capture_output=True, text=True, timeout=timeout, input=inp, errors=\"replace\")\n        return p.returncode, (p.stdout or \"\") + (p.stderr or \"\"), time.time() - t0\n    except subprocess.TimeoutExpired as e:\n        return -999, ((e.stdout or b\"\").decode(\"utf-8\", \"replace\") if isinstance(e.stdout, bytes) else (e.stdout or \"\")) + \"\\n[timeout]\", time.time() - t0\n\n\ndef _tail(s, n=900):\n    return s[-n:] if s else \"\"\n\n\ndef _statuses(out):\n    return {m.group(2).strip(): m.group(1) for m in STATUS.finditer(out)}\n\n\ndef run_task(task, work_root, keep=False):\n    iid, repo = task[\"instance_id\"], task[\"repo\"]\n    d = Path(work_root) / iid\n    shutil.rmtree(d, ignore_errors=True)\n    d.mkdir(parents=True)\n    rdir, vdir = d / \"repo\", d / \"venv\"\n    res = {\"id\": iid, \"repo\": repo, \"python_requested\": (task.get(\"install_config\") or {}).get(\"python\"), \"times\": {}}\n    binname = \"Scripts\" if os.name == \"nt\" else \"bin\"\n    env = dict(os.environ)\n    env.update({\"PIP_DISABLE_PIP_VERSION_CHECK\": \"1\", \"GIT_TERMINAL_PROMPT\": \"0\", \"PYTHONDONTWRITEBYTECODE\": \"1\", \"PYTHONNOUSERSITE\": \"1\"})\n    env.pop(\"PYTHONPATH\", None)   # Kaggle's PYTHONPATH carries a sitecustomize that imports `wrapt`, which breaks clean venvs and pip's isolated build envs\n    for k, v in ((task.get(\"install_config\") or {}).get(\"env_vars\") or {}).items():\n        if v is not None:\n            env[k] = str(v)\n    try:\n        # ---- clone at base_commit\n        rdir.mkdir()\n        for cmd in ([\"git\", \"init\", \"-q\"], [\"git\", \"remote\", \"add\", \"origin\", f\"https://github.com/{repo}.git\"]):\n            _run(cmd, rdir, 120, env)\n        rc, out, dt = _run([\"git\", \"fetch\", \"-q\", \"--depth\", \"1\", \"origin\", task[\"base_commit\"]], rdir, 900, env)\n        if rc != 0:\n            rc, out, dt2 = _run([\"git\", \"fetch\", \"-q\", \"origin\", task[\"base_commit\"]], rdir, 1500, env)\n            dt += dt2\n        if rc == 0:\n            rc, out2, dt3 = _run([\"git\", \"checkout\", \"-q\", \"FETCH_HEAD\"], rdir, 300, env)\n            out += out2\n            dt += dt3\n        res[\"times\"][\"clone\"] = round(dt)\n        if rc != 0:\n            res.update(verdict=\"clone_failed\", reason=_tail(out, 300))\n            return res\n        # ---- venv + install\n        # Kaggle's image has a broken `ensurepip` (python -m venv fails), so use virtualenv, which brings its own seed wheels\n        rc, out, dt = (1, \"forced\", 0) if os.environ.get(\"G7_FORCE_VIRTUALENV\") else _run([sys.executable, \"-m\", \"venv\", str(vdir)], d, 300, env)\n        if rc != 0:\n            shutil.rmtree(vdir, ignore_errors=True)\n            rc, out, dt = _run([sys.executable, \"-m\", \"virtualenv\", \"-q\", str(vdir)], d, 300, env)\n            if rc != 0 and \"No module named virtualenv\" in out:\n                _run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"virtualenv\"], d, 300, env)\n                rc, out, dt = _run([sys.executable, \"-m\", \"virtualenv\", \"-q\", str(vdir)], d, 300, env)\n        if rc != 0:\n            res.update(verdict=\"install_failed\", reason=\"venv/virtualenv: \" + _tail(out, 300))\n            return res\n        venv_bin = vdir / binname\n        env[\"VIRTUAL_ENV\"] = str(vdir)\n        env[\"PATH\"] = str(venv_bin) + os.pathsep + env.get(\"PATH\", \"\")\n        vpy = str(venv_bin / (\"python.exe\" if os.name == \"nt\" else \"python\"))\n        rc, out, dt = _run([vpy, \"-m\", \"pip\", \"install\", \"-q\", \"pytest\", \"pytest-timeout\", \"wrapt\"], rdir, 600, env)\n        ic = task.get(\"install_config\") or {}\n        steps = list(ic.get(\"pre_install\") or []) + ([ic[\"install\"]] if ic.get(\"install\") else [\"pip install -e .\"])\n        install_out = \"\"\n        for s in steps:\n            rc, o, dt2 = _run(s, rdir, 1500, env, shell=True)\n            dt += dt2\n            install_out += f\"$ {s}\\n{_tail(o, 600)}\\n\"\n            if rc != 0:\n                res[\"times\"][\"install\"] = round(dt)\n                res.update(verdict=\"install_failed\", reason=f\"`{s}` rc={rc}: {_tail(o, 400)}\")\n                return res\n        pkgs = [x for x in (ic.get(\"pip_packages\") or []) if x]\n        if pkgs:\n            rc, o, dt2 = _run([vpy, \"-m\", \"pip\", \"install\", \"-q\", *pkgs], rdir, 900, env)\n            dt += dt2\n            res[\"pip_packages\"] = {\"n\": len(pkgs), \"rc\": rc}\n        res[\"times\"][\"install\"] = round(dt)\n        rc, out, _ = _run([vpy, \"-m\", \"pip\", \"freeze\"], rdir, 120, env)\n        res[\"n_packages\"] = len([l for l in out.splitlines() if l.strip()])\n        res[\"freeze\"] = out.splitlines()[:400]\n        # ---- tests before (test patch only)\n        for name, patch in ((\"test_patch\", task[\"test_patch\"]),):\n            rc, o, _ = _run([\"git\", \"apply\", \"--whitespace=nowarn\", \"-\"], rdir, 120, env, inp=patch)\n            if rc != 0:\n                res.update(verdict=\"patch_apply_failed\", reason=f\"{name}: {_tail(o, 300)}\")\n                return res\n        f2p, p2p = list(task[\"FAIL_TO_PASS\"]), list(task[\"PASS_TO_PASS\"])[:150]\n        base = [vpy, \"-m\", \"pytest\", \"-rA\", \"--tb=line\", \"-q\", \"-p\", \"no:cacheprovider\", \"--timeout=300\"]\n        added = []\n\n        def run_with_deps(ids, timeout):\n            \"\"\"Run pytest; if collection dies on `No module named X`, pip install X and retry (max 4 times). This mimics a shared wheel set that holds test deps.\"\"\"\n            for _ in range(5):\n                rc_, o_, dt_ = _run(base + ids, rdir, timeout, env)\n                miss = re.findall(r\"No module named '([A-Za-z0-9_\\-]+)\", o_)\n                miss = [m for m in dict.fromkeys(miss) if m not in added and m.split(\".\")[0] not in added]\n                if rc_ == -999 or not miss or len(added) >= 8:\n                    return rc_, o_, dt_\n                for m in miss[:3]:\n                    added.append(m)\n                    _run([vpy, \"-m\", \"pip\", \"install\", \"-q\", ALIASES.get(m, m)], rdir, 300, env)\n            return rc_, o_, dt_\n\n        rc, out_b, dt = run_with_deps(f2p, 1200)\n        res[\"times\"][\"tests_before\"] = round(dt)\n        sb = _statuses(out_b)\n        if rc == -999:\n            res.update(verdict=\"timeout\", reason=\"tests before the patch timed out\")\n            return res\n        if not any(i in sb for i in f2p) and not sb:\n            res.update(verdict=\"no_tests_ran\", reason=_tail(out_b, 400))\n            return res\n        passed_before = [i for i in f2p if sb.get(i) == \"PASSED\"]\n        # ---- gold patch, tests after\n        rc, o, _ = _run([\"git\", \"apply\", \"--whitespace=nowarn\", \"-\"], rdir, 120, env, inp=task[\"patch\"])\n        if rc != 0:\n            res.update(verdict=\"patch_apply_failed\", reason=\"gold patch: \" + _tail(o, 300))\n            return res\n        rc, out_a, dt = run_with_deps(f2p + p2p, 1800)\n        res[\"extra_modules_installed\"] = list(added)\n        res[\"times\"][\"tests_after\"] = round(dt)\n        sa = _statuses(out_a)\n        if rc == -999:\n            res.update(verdict=\"timeout\", reason=\"tests after the patch timed out\")\n            return res\n        f2p_ok = [i for i in f2p if sa.get(i) == \"PASSED\"]\n        p2p_ok = [i for i in p2p if sa.get(i) == \"PASSED\"]\n        res.update(f2p=len(f2p), f2p_pass_before=len(passed_before), f2p_pass_after=len(f2p_ok), p2p=len(p2p), p2p_pass_after=len(p2p_ok))\n        if passed_before:\n            res.update(verdict=\"f2p_passes_before\", reason=f\"{len(passed_before)}/{len(f2p)} F2P already pass with only the test patch\")\n        elif len(f2p_ok) != len(f2p):\n            bad = [(i, sa.get(i)) for i in f2p if sa.get(i) != \"PASSED\"][:3]\n            errs = [l for l in out_a.splitlines() if (\"Error\" in l or \"error\" in l) and \"PASSED\" not in l][:4]\n            res.update(verdict=\"f2p_fails_after\", reason=f\"{bad} | {' / '.join(errs)[:500]} | {_tail(out_a, 200)}\")\n        elif len(p2p_ok) != len(p2p):\n            res.update(verdict=\"p2p_partial\", reason=f\"{len(p2p) - len(p2p_ok)}/{len(p2p)} P2P not passing after the gold patch\")\n        else:\n            res.update(verdict=\"valid\", reason=\"\")\n        return res\n    except Exception as e:  # never let one task kill the spike\n        res.update(verdict=\"harness_error\", reason=repr(e)[:300])\n        return res\n    finally:\n        if not keep:\n            shutil.rmtree(d, ignore_errors=True)\n"
_p = next(Path('/kaggle/input').rglob('candidates.jsonl'))
_all = [json.loads(l) for l in open(_p, encoding='utf-8')]
TASKS = [t for i, t in enumerate(_all) if i % N_SHARDS == SHARD]
print(len(_all), 'candidates;', len(TASKS), 'in this shard')

In [ ]:
import json, os, time, subprocess, sys, collections, concurrent.futures as cf
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], check=True)   # Kaggle's `python -m venv` is broken (ensurepip)
exec(WORKER_SRC)   # defines run_task(task, work_root)
WORK = Path('/tmp/scale'); WORK.mkdir(exist_ok=True)
OUTDIR = Path('/kaggle/working'); OUTDIR.mkdir(exist_ok=True)
RES_PATH = OUTDIR / f'scale_results_{SHARD}.json'
BUDGET_S = BUDGET_H * 3600
t0 = time.time(); results = []
def _one(t):
    if time.time() - t0 > BUDGET_S: return {'id': t['instance_id'], 'repo': t['repo'], 'verdict': 'skipped_budget', 'reason': ''}
    try: r = run_task(t, WORK)
    except Exception as e: r = {'id': t['instance_id'], 'repo': t['repo'], 'verdict': 'harness_error', 'reason': repr(e)[:300]}
    if r.get('verdict') not in ('valid', 'p2p_partial'): r.pop('freeze', None)
    return r
with cf.ThreadPoolExecutor(max_workers=4) as ex:
    futs = [ex.submit(_one, t) for t in TASKS]
    for k, f in enumerate(cf.as_completed(futs), 1):
        r = f.result(); results.append(r)
        if k % 10 == 0 or k == len(futs):
            c = collections.Counter(x['verdict'] for x in results)
            print(f"[{time.time() - t0:7.0f}s] {k}/{len(futs)} done | " + ' '.join(f'{a}={b}' for a, b in c.most_common()), flush=True)
            RES_PATH.write_text(json.dumps(results))
RES_PATH.write_text(json.dumps(results))
print('\nverdicts:', dict(collections.Counter(r['verdict'] for r in results)), '| wall', round(time.time() - t0), 's')
fails = collections.Counter((r['verdict'], (r.get('reason') or '')[:60]) for r in results if r['verdict'] not in ('valid',))
for (v, why), n in fails.most_common(25): print(f'  {n:3d} {v:17s} {why!r}')